# Tutorial 4: subsystem and graph artifacts

This notebook ranks subsystems from the installed comparison artifact, builds renderer-independent Graph Artifacts, and renders those artifacts through the Bokeh adapter.


## Before you run it

The notebook is self-contained and repeats the loading, grouping, and comparison setup so that the graph visualizations can be explored independently of the earlier notebooks.

In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

from bokeh.io import output_notebook, show

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    AnalysisWorkspace,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    GroupComparison,
    GroupComparisonSpec,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    plot_subsystem_comparison,
)
from metabis.analysis_workspace import RevisionKind
from metabis.version import __version__ as metabis_version
from metabis.viz.metabolic_visualization import (
    EnrichmentConfig,
    GraphGenerationConfig,
    MetabolicGraphPipeline,
    PipelineConfig,
    RenderingConfig,
    render_graph_artifact,
)

output_notebook()


In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = repo_root / "data"
required_example_files = (
    "Th17_Compass_reactions.tsv",
    "Th17_Compass_expression.tsv",
    "Th17_Compass_metadata.tsv",
)
missing = [name for name in required_example_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing example files: "
        + ", ".join(missing)
        + ". Run `python data/download_datasets.py`."
    )

reaction_data_path = data_dir / "Th17_Compass_reactions.tsv"
expression_data_path = data_dir / "Th17_Compass_expression.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
target_group_id = "Th17p"
baseline_group_id = "Th17n"

model = MetabolicModel(use_cache=True)
model.apply_alignment_corrections(model.get_alignment_file_path("RECON2"))

reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    data_mode="dual",
    metabolic_model=model,
    reaction_name_format="rxnRecon3DID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(compass_activity=True),
)
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    gene_format="geneSymbols",
    preprocessing=ExpressionPreprocessingConfig(normalize_total=True),
    data_format="csv",
    csv_separator="\t",
    samples_in="columns",
)
metadata = SampleMetadata.from_table(
    metadata_path,
    sample_id_column="sample",
)

dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
dm.update_sample_metadata(metadata)

print(f"Metabis version: {metabis_version}")
print(f"Loaded {len(reaction_data.get_sample_ids())} aligned samples")
grouping_result = dm.load_groups_from_metadata(["cell_type"])
print(f"Groups available for comparison: {dm.get_group_ids()}")
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)

analysis = ExperimentAnalysis(dm)
dm.install_experiment_analysis(analysis_snapshot.provenance, analysis)
dm.refresh_experiment_analysis_presentation()
dm.set_analysis_group(target_group_id)

comparison_spec = GroupComparisonSpec.pairwise(
    (target_group_id,),
    (baseline_group_id,),
    target_label="Pathogenic Th17",
    baseline_label="Non-pathogenic Th17",
)
group_comparison = GroupComparison(
    spec=comparison_spec,
    test="wilcoxon",
    correction="fdr_bh",
)
group_comparison.run_test(dm, for_genes=False)
group_comparison.run_test(dm, for_genes=True)

membership = group_comparison.membership
assert membership is not None
print(
    f"Target: {membership.target_label} ({len(membership.target_samples)} samples)"
)
print(
    f"Baseline: {membership.baseline_label} "
    f"({len(membership.baseline_samples)} samples)"
)

analysis_settings = {
    "pval_threshold": 0.10,
    "cohens_d_threshold": 0.0,
    "log2_fc_threshold": 0.0,
    "mean_difference_threshold": 0.0,
    "keep_reversed": True,
}
group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=False,
)
group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=True,
)

comparison_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
comparison_artifact = group_comparison.build_artifact(
    provenance=comparison_snapshot.provenance,
    source_fingerprints=dict(dm.source_fingerprints),
    keep_reversed=analysis_settings["keep_reversed"],
)
dm.install_group_comparison_artifact(
    comparison_snapshot.provenance,
    comparison_artifact,
)
dm.refresh_group_comparison_presentation()

print(f"Current mode: {dm.get_mode()}")
print(
    "Significant reactions available for graph coloring: "
    f"{len(comparison_artifact.significant_reactions)}"
)

def show_graph_artifact(
    element_name,
    graph_type,
    *,
    layout_engine,
    height,
):
    graph_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
    pipeline = MetabolicGraphPipeline(
        model,
        PipelineConfig(
            generation_config=GraphGenerationConfig(
                use_human1_map=True,
                layout_engine=layout_engine,
            ),
            enrichment_config=EnrichmentConfig(color_metric="cohens_d"),
            show_plot=False,
            enable_timing=True,
        ),
    )
    artifact = pipeline.build_artifact(
        element_name,
        graph_type,
        data_manager=dm,
        provenance=graph_snapshot.provenance,
        source_fingerprints=dict(dm.source_fingerprints),
    )
    rendered = render_graph_artifact(
        artifact,
        RenderingConfig(
            height=height,
            sizing_mode="scale_both",
            aspect_ratio=2,
            zoom=1,
        ),
    )
    show(rendered.plot)
    print(
        f"Rendered {artifact.display_title}: "
        f"{artifact.node_count} nodes, {artifact.edge_count} edges"
    )
    return artifact


## 1. Rank subsystems from the comparison

`SubsystemAnalysis` lives inside the `AnalysisWorkspace`. It converts the current comparison into subsystem-level scores using a configurable delta definition.

The default configuration is already sensible, but exposing it explicitly makes the object easier to understand.

In [ ]:
delta_options = {
    "metric": "cohens_d",
    "transform": "sign",
    "numerator_filter": "significant",
    "denominator_filter": "all",
}

dm.subsystem_analysis.update_delta_config(delta_options)
subsystem_stats = dm.subsystem_analysis.rank_subsystems_by_delta_score(for_genes=False)

subsystem_stats.head(10)

## 2. Plot subsystem categories

`plot_subsystem_comparison()` groups subsystems by metabolic category and ranks them by the chosen subsystem-level score. It is a compact way to decide which pathways are worth opening as graphs.

In [ ]:
plot_subsystem_comparison(
    dm,
    subsystem_stats=subsystem_stats,
    metric="cohens_d",
    width=950,
    return_plot=False,
)

## 3. Build and render the full-network Graph Artifact

`MetabolicGraphPipeline.build_artifact()` performs graph analysis without constructing Bokeh models. `render_graph_artifact()` is the separate presentation adapter.


In [ ]:
overview_artifact = show_graph_artifact(
    None,
    "metabolic_network",
    layout_engine="fdp",
    height=850,
)


## 4. Open a detailed subsystem graph

Once the ranking plot tells you where to look, the same plotting function can open one subsystem at a time.

Here we reuse the TCA-cycle example from the main tutorial because it is a stable, interpretable subsystem for this dataset.

In [ ]:
selected_subsystem = "Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism"
selected_subsystem_stats = dm.subsystem_analysis.get_subsystem_statistics(selected_subsystem)

print(
    f"{selected_subsystem}: "
    f"{selected_subsystem_stats['significant_elements']}/{selected_subsystem_stats['total_elements']} significant reactions, "
    f"delta subsystem score = {selected_subsystem_stats['delta_subsystem_score']:.3f}"
 )

In [ ]:
tca_artifact = show_graph_artifact(
    selected_subsystem,
    "subsystem",
    layout_engine="dot",
    height=750,
)


## Optional second subsystem

The same API can be reused immediately for another subsystem. The arginine and proline pathway is a useful follow-up because it connects reaction-level changes to familiar gene-level biology in this dataset.

In [ ]:
arginine_artifact = show_graph_artifact(
    "Arginine and proline metabolism",
    "subsystem",
    layout_engine="dot",
    height=850,
)


## Takeaways

Scientific graph analysis now ends in plain immutable Graph Artifacts. Bokeh rendering is an adapter over those artifacts, so exports and plots share exactly the same graph structure, colors, labels, and provenance.
